# 11. External Field Confirmatory Evaluation & 224×224 Resolution Hypothesis Test

This notebook executes the final confirmatory phase of the project without updating any model weights:
1. **Leakage & Duplicate Audit (`tomato_field_eval_v1`, $n=62$):** Verifies SHA-256 and 64-bit DCT pHash ($d_{\text{pHash}} \le 4$) independence against all 523 PlantDoc images.
2. **Confirmatory Evaluation of the 4 Locked Models:** Evaluates `ResNet18 · Lab baseline`, `ViT-Small · Lab baseline`, `ResNet18 · Geometric + Mixed`, and `ViT-Small · Mixed` on:
   - **External Field Eval v1 ($n=62$):** Overall and stratified by `source` (`self_collected` $n=16$ vs. `online_sources` $n=46$), `visible_leaf_context` (`multiple_leaves` $n=47$ vs. `single_or_partial_leaf` $n=15$), and `image_type` (`multi_leaf` $n=39$, `cropped` $n=18$, `single_leaf` $n=5$).
   - **Original PlantDoc Field Test ($n=47$):** Complete 4-model comparison.
3. **Controlled $224\times 224$ Resolution & Cropping Experiment on *Bacterial Spot*:** Tests whether full-frame downsampling to $224\times 224$ destroys fine-grained lesion details by comparing `1.00x Full-Scene`, `0.70x Center Leaf-ROI Crop`, `0.50x Close-Up Lesion Crop`, and `448×448 High-Res 5-Crop Tiling` across both `tomato_field_eval_v1` ($n=62$) and `Verified Field Dev` ($n=92$).

In [ ]:
%pip install -q "timm==1.0.26" datasets pandas scikit-learn matplotlib pillow imagehash scipy

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import timm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

REPO_URL = "https://github.com/thutrang147/plant-disease-diagnosis.git"
BRANCH = "feat/statistical-analysis-and-field-eval"
PROJECT_ROOT = Path("/content/plant-disease-diagnosis")
if PROJECT_ROOT.exists():
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=True)

sys.path.insert(0, str(PROJECT_ROOT))
from src.dataset import PlantDiseaseDataset
from src.evaluate_confirmatory_field import audit_external_dataset
from src.statistical_analysis import bootstrap_binomial_ci, compute_ece, mcnemar_exact, wilson_ci

from google.colab import drive, files
drive.mount("/content/drive")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external_evaluation" / "tomato_field_eval_v1" / "tomato_field_eval_v1"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "external_confirmatory_eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Evaluation device:", DEVICE)
print("External dataset dir exists:", EXTERNAL_DIR.is_dir())
print("Output directory:", OUTPUT_DIR)

In [ ]:
# Automatically locate the plant-disease-diagnosis folder in Google Drive (MyDrive or Shortcut)
CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/plant-disease-diagnosis"),
    Path("/content/drive/Shareddrives/plant-disease-diagnosis"),
]
for p in Path("/content/drive/MyDrive").glob("**/resnet18_lab_best.pt"):
    CANDIDATE_ROOTS.insert(0, p.parent.parent)
    break

DRIVE_ROOT = None
for cand in CANDIDATE_ROOTS:
    if (cand / "resnet18_baseline" / "resnet18_lab_best.pt").is_file():
        DRIVE_ROOT = cand
        break

if DRIVE_ROOT is None:
    raise FileNotFoundError(
        "Could not find 'plant-disease-diagnosis' in /content/drive/MyDrive!\n"
        "Please right-click the shared 'plant-disease-diagnosis' folder in Google Drive -> "
        "Sắp xếp (Organize) -> Thêm lối tắt (Add shortcut) -> Drive của tôi (My Drive), then re-run this cell."
    )

print("Found Google Drive checkpoint root at:", DRIVE_ROOT)

LABEL_MAP = json.loads((SPLITS_DIR / "label_map.json").read_text(encoding="utf-8"))
CLASS_NAMES = [name for name, idx in sorted(LABEL_MAP.items(), key=lambda item: item[1])]
LABEL_IDS = list(range(len(CLASS_NAMES)))

CHECKPOINT_SPECS = {
    "ResNet18 · Lab baseline": {
        "short": "resnet18_lab_base",
        "architecture": "resnet18",
        "checkpoint": DRIVE_ROOT / "resnet18_baseline/resnet18_lab_best.pt",
        "source_checkpoint": DRIVE_ROOT / "resnet18_baseline/resnet18_lab_best.pt",
        "batch_size": 32,
    },
    "ViT-Small · Lab baseline": {
        "short": "vit_small_lab_base",
        "architecture": "vit_small_patch16_224.augreg_in1k",
        "checkpoint": DRIVE_ROOT / "vit_small_baseline/vit_small_lab_best.pt",
        "source_checkpoint": DRIVE_ROOT / "vit_small_baseline/vit_small_lab_best.pt",
        "batch_size": 16,
    },
    "ResNet18 · Geometric + Mixed": {
        "short": "resnet18_geom_mixed",
        "architecture": "resnet18",
        "checkpoint": DRIVE_ROOT / "mixed_domain_verified/R2_resnet_geometric/mixed_domain_best.pt",
        "source_checkpoint": DRIVE_ROOT / "resnet18_aug_geometric/resnet18_best.pt",
        "batch_size": 32,
    },
    "ViT-Small · Mixed": {
        "short": "vit_small_mixed",
        "architecture": "vit_small_patch16_224.augreg_in1k",
        "checkpoint": DRIVE_ROOT / "mixed_domain_verified/V1_vit_small/mixed_domain_best.pt",
        "source_checkpoint": DRIVE_ROOT / "vit_small_baseline/vit_small_lab_best.pt",
        "batch_size": 16,
    },
}

for name, spec in CHECKPOINT_SPECS.items():
    assert spec["checkpoint"].is_file(), f"Missing checkpoint: {spec['checkpoint']}"
    assert spec["source_checkpoint"].is_file(), f"Missing source checkpoint: {spec['source_checkpoint']}"
print("All 4 primary checkpoints verified!")

In [ ]:
# 1. Run pHash and exact SHA-256 audit on tomato_field_eval_v1 (n=62)
audited_ext_df, audit_summary = audit_external_dataset(
    external_dir=EXTERNAL_DIR,
    splits_dir=SPLITS_DIR,
    phash_threshold=4,
    output_dir=OUTPUT_DIR,
)
print(json.dumps(audit_summary, indent=2))

In [ ]:
class ExternalManifestDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: Path, transform=None, center_crop_fraction: float = 1.0):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.center_crop_fraction = float(center_crop_fraction)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = self.root_dir / row["image_path"]
        with Image.open(img_path) as im:
            rgb = im.convert("RGB")
        if self.center_crop_fraction < 0.999:
            w, h = rgb.size
            cw, ch = max(16, int(round(w * self.center_crop_fraction))), max(16, int(round(h * self.center_crop_fraction)))
            left = (w - cw) // 2
            top = (h - ch) // 2
            rgb = rgb.crop((left, top, left + cw, top + ch))
        if self.transform is not None:
            tensor = self.transform(rgb)
        else:
            tensor = rgb
        return tensor, int(row["label_id"])


class CroppedPlantDiseaseDataset(Dataset):
    """Wraps PlantDiseaseDataset with an optional pre-resize center ROI crop fraction."""
    def __init__(self, base_ds: PlantDiseaseDataset, transform, center_crop_fraction: float = 1.0):
        self.base_ds = base_ds
        self.transform = transform
        self.center_crop_fraction = float(center_crop_fraction)

    def __len__(self):
        return len(self.base_ds)

    def __getitem__(self, idx: int):
        row = self.base_ds.metadata.iloc[idx]
        source = row["source"]
        row_id = int(row["row_id"])
        record = self.base_ds.source_datasets[source][row_id]
        rgb = record["image"].convert("RGB")
        if self.center_crop_fraction < 0.999:
            w, h = rgb.size
            cw, ch = max(16, int(round(w * self.center_crop_fraction))), max(16, int(round(h * self.center_crop_fraction)))
            left = (w - cw) // 2
            top = (h - ch) // 2
            rgb = rgb.crop((left, top, left + cw, top + ch))
        return self.transform(rgb), int(row["label_id"])


def load_fixed_model(name: str):
    spec = CHECKPOINT_SPECS[name]
    state = torch.load(spec["checkpoint"], map_location="cpu", weights_only=True)
    source = torch.load(spec["source_checkpoint"], map_location="cpu", weights_only=True)
    architecture = spec["architecture"]
    if architecture == "resnet18":
        model = resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
        transform = ResNet18_Weights.IMAGENET1K_V1.transforms()
        norm_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        config = source["preprocessing_config"]
        model = timm.create_model(architecture, pretrained=False, num_classes=len(CLASS_NAMES))
        transform = timm.data.create_transform(**config, is_training=False)
        norm_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=config.get("mean", (0.5, 0.5, 0.5)), std=config.get("std", (0.5, 0.5, 0.5))),
        ])
    model.load_state_dict(state["model_state_dict"], strict=True)
    model.to(DEVICE).eval()
    return model, transform, norm_tf


def run_inference(model, loader) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    y_true, y_pred, conf, all_probs = [], [], [], []
    with torch.inference_mode():
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            probs = torch.softmax(logits.float(), dim=1)
            p_max, pred = probs.max(dim=1)
            y_true.extend(labels.tolist())
            y_pred.extend(pred.cpu().tolist())
            conf.extend(p_max.cpu().tolist())
            all_probs.append(probs.cpu().numpy())
    return (
        np.asarray(y_true, dtype=int),
        np.asarray(y_pred, dtype=int),
        np.asarray(conf, dtype=float),
        np.vstack(all_probs),
    )


def summarize_predictions(model_name: str, partition_name: str, y_true: np.ndarray, y_pred: np.ndarray, conf: np.ndarray) -> dict:
    corr = (y_true == y_pred)
    n = int(len(y_true))
    k = int(np.sum(corr))
    acc = float(accuracy_score(y_true, y_pred))
    mf1 = float(f1_score(y_true, y_pred, labels=LABEL_IDS, average="macro", zero_division=0))
    w_lo, w_hi = wilson_ci(k, n)
    b_lo, b_hi = bootstrap_binomial_ci(k, n)
    ece = compute_ece(corr, conf, n_bins=10)
    errs = ~corr
    n_err = int(np.sum(errs))
    hi_err = int(np.sum(errs & (conf >= 0.90)))
    return {
        "model": model_name,
        "partition": partition_name,
        "n": n,
        "correct": k,
        "accuracy": round(acc, 4),
        "wilson_95ci_low": round(w_lo, 4),
        "wilson_95ci_high": round(w_hi, 4),
        "bootstrap_95ci_low": round(b_lo, 4),
        "bootstrap_95ci_high": round(b_hi, 4),
        "macro_f1": round(mf1, 4),
        "ece_10bins": round(ece, 4),
        "mean_confidence": round(float(np.mean(conf)), 4),
        "mean_conf_correct": round(float(np.mean(conf[corr])), 4) if k > 0 else np.nan,
        "mean_conf_error": round(float(np.mean(conf[errs])), 4) if n_err > 0 else np.nan,
        "total_errors": n_err,
        "high_conf_errors_ge_0_90": hi_err,
        "high_conf_share_of_errors": round(hi_err / n_err, 4) if n_err > 0 else 0.0,
    }

In [ ]:
# 2. Evaluate all 4 locked models on External Field Eval v1 (n=62) and PlantDoc Field Test (n=47)
LOADED_MODELS = {name: load_fixed_model(name) for name in CHECKPOINT_SPECS}

summary_rows = []
per_class_rows = []
subgroup_rows = []
ext_predictions = {}
ftest_predictions = {}

base_field_test_ds = PlantDiseaseDataset(SPLITS_DIR / "field_test.csv", transform=None)
base_field_dev_ds = PlantDiseaseDataset(SPLITS_DIR / "field_dev.csv", transform=None)

for idx, (name, (model, tf, _)) in enumerate(LOADED_MODELS.items(), start=1):
    short = CHECKPOINT_SPECS[name]["short"]
    bs = CHECKPOINT_SPECS[name]["batch_size"]

    # A) External Field Eval v1 (n=62)
    ext_ds = ExternalManifestDataset(audited_ext_df, EXTERNAL_DIR, transform=tf, center_crop_fraction=1.0)
    ext_loader = DataLoader(ext_ds, batch_size=bs, shuffle=False)
    y_true, y_pred, conf, probs = run_inference(model, ext_loader)
    ext_predictions[name] = (y_true, y_pred, conf, probs)
    summary_rows.append(summarize_predictions(name, "external_field_v1", y_true, y_pred, conf))

    pred_df = audited_ext_df.copy()
    pred_df["predicted_label_id"] = y_pred
    pred_df["predicted_class"] = [CLASS_NAMES[i] for i in y_pred]
    pred_df["confidence"] = np.round(conf, 6)
    pred_df["correct"] = (y_true == y_pred)
    for c_id, c_name in enumerate(CLASS_NAMES):
        pred_df[f"prob_{c_name.lower().replace(' ', '_')}"] = np.round(probs[:, c_id], 6)
    pred_df.to_csv(OUTPUT_DIR / f"predictions_model{idx}_{short}_external_v1.csv", index=False)

    prec, rec, f1s, sup = precision_recall_fscore_support(y_true, y_pred, labels=LABEL_IDS, zero_division=0)
    for c_id, c_name in enumerate(CLASS_NAMES):
        c_mask = (y_true == c_id)
        c_k = int(np.sum((y_pred == c_id) & c_mask))
        c_n = int(np.sum(c_mask))
        w_lo, w_hi = wilson_ci(c_k, c_n)
        per_class_rows.append({
            "model": name,
            "partition": "external_field_v1",
            "label_id": c_id,
            "class_name": c_name,
            "support": c_n,
            "correct": c_k,
            "precision": round(float(prec[c_id]), 4),
            "recall": round(float(rec[c_id]), 4),
            "recall_wilson_95ci_low": round(w_lo, 4),
            "recall_wilson_95ci_high": round(w_hi, 4),
            "f1": round(float(f1s[c_id]), 4),
        })

    # Subgroups on External Field Eval v1
    for col in ("source", "visible_leaf_context", "image_type"):
        for val, sub_df in pred_df.groupby(col):
            st = sub_df["label_id"].to_numpy(dtype=int)
            sp = sub_df["predicted_label_id"].to_numpy(dtype=int)
            sc = sub_df["confidence"].to_numpy(dtype=float)
            present_labels = sorted(np.unique(st).tolist())
            w_lo, w_hi = wilson_ci(int(np.sum(st == sp)), len(st))
            subgroup_rows.append({
                "model": name,
                "stratum_column": col,
                "stratum_value": str(val),
                "n": int(len(st)),
                "correct": int(np.sum(st == sp)),
                "accuracy": round(float(accuracy_score(st, sp)), 4),
                "wilson_95ci_low": round(w_lo, 4),
                "wilson_95ci_high": round(w_hi, 4),
                "macro_f1_present_classes": round(float(f1_score(st, sp, labels=present_labels, average="macro", zero_division=0)), 4),
            })

    # B) Original PlantDoc Field Test (n=47)
    ft_ds = CroppedPlantDiseaseDataset(base_field_test_ds, transform=tf, center_crop_fraction=1.0)
    ft_loader = DataLoader(ft_ds, batch_size=bs, shuffle=False)
    ft_true, ft_pred, ft_conf, ft_probs = run_inference(model, ft_loader)
    ftest_predictions[name] = (ft_true, ft_pred, ft_conf, ft_probs)
    summary_rows.append(summarize_predictions(name, "plantdoc_field_test", ft_true, ft_pred, ft_conf))

    ft_prec, ft_rec, ft_f1s, _ = precision_recall_fscore_support(ft_true, ft_pred, labels=LABEL_IDS, zero_division=0)
    for c_id, c_name in enumerate(CLASS_NAMES):
        c_mask = (ft_true == c_id)
        c_k = int(np.sum((ft_pred == c_id) & c_mask))
        c_n = int(np.sum(c_mask))
        w_lo, w_hi = wilson_ci(c_k, c_n)
        per_class_rows.append({
            "model": name,
            "partition": "plantdoc_field_test",
            "label_id": c_id,
            "class_name": c_name,
            "support": c_n,
            "correct": c_k,
            "precision": round(float(ft_prec[c_id]), 4),
            "recall": round(float(ft_rec[c_id]), 4),
            "recall_wilson_95ci_low": round(w_lo, 4),
            "recall_wilson_95ci_high": round(w_hi, 4),
            "f1": round(float(ft_f1s[c_id]), 4),
        })

summary_df = pd.DataFrame(summary_rows)
per_class_df = pd.DataFrame(per_class_rows)
subgroup_df = pd.DataFrame(subgroup_rows)

summary_df.to_csv(OUTPUT_DIR / "confirmatory_summary_all_partitions.csv", index=False)
per_class_df.to_csv(OUTPUT_DIR / "confirmatory_per_class_metrics.csv", index=False)
subgroup_df.to_csv(OUTPUT_DIR / "external_v1_subgroup_breakdown.csv", index=False)

print("=== Confirmatory Summary (External Field v1 & PlantDoc Field Test) ===")
display(summary_df)
print("\n=== External Field v1 Per-Class Recall ===")
display(per_class_df[per_class_df["partition"] == "external_field_v1"].pivot(index="class_name", columns="model", values="recall"))
print("\n=== External Field v1 Subgroup Breakdown ===")
display(subgroup_df)

In [ ]:
# 3. Paired McNemar Tests and Class-Wise Paired Outcomes on External Field Eval v1 (n=62)
mcnemar_rows = []
for part_name, pred_dict in (("external_field_v1", ext_predictions), ("plantdoc_field_test", ftest_predictions)):
    pairs_to_test = [
        ("ResNet18 · Geometric + Mixed", "ResNet18 · Lab baseline"),
        ("ViT-Small · Mixed", "ViT-Small · Lab baseline"),
        ("ResNet18 · Geometric + Mixed", "ViT-Small · Mixed"),
        ("ViT-Small · Lab baseline", "ResNet18 · Lab baseline"),
    ]
    for m_a, m_b in pairs_to_test:
        yt_a, yp_a, _, _ = pred_dict[m_a]
        yt_b, yp_b, _, _ = pred_dict[m_b]
        ca = (yt_a == yp_a)
        cb = (yt_b == yp_b)
        both_c = int(np.sum(ca & cb))
        only_a = int(np.sum(ca & (~cb)))
        only_b = int(np.sum((~ca) & cb))
        both_w = int(np.sum((~ca) & (~cb)))
        pval = mcnemar_exact(only_a, only_b)
        mcnemar_rows.append({
            "partition": part_name,
            "model_a": m_a,
            "model_b": m_b,
            "n": int(len(ca)),
            "both_correct": both_c,
            "only_a_correct": only_a,
            "only_b_correct": only_b,
            "both_wrong": both_w,
            "exact_mcnemar_p": round(float(pval), 6),
        })

mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv(OUTPUT_DIR / "confirmatory_paired_mcnemar_tests.csv", index=False)
display(mcnemar_df)

# Class-wise paired table on External Field v1 between ResNet18 Geom+Mixed and ViT-Small Mixed
yt, yp_r, _, _ = ext_predictions["ResNet18 · Geometric + Mixed"]
_, yp_v, _, _ = ext_predictions["ViT-Small · Mixed"]
rc = (yt == yp_r)
vc = (yt == yp_v)
paired_class_rows = []
for c_id, c_name in enumerate(CLASS_NAMES):
    m = (yt == c_id)
    paired_class_rows.append({
        "label_id": c_id,
        "class_name": c_name,
        "support": int(np.sum(m)),
        "both_correct": int(np.sum(m & rc & vc)),
        "resnet_only_correct": int(np.sum(m & rc & (~vc))),
        "vit_only_correct": int(np.sum(m & (~rc) & vc)),
        "both_wrong": int(np.sum(m & (~rc) & (~vc))),
    })
paired_class_df = pd.DataFrame(paired_class_rows)
paired_class_df.to_csv(OUTPUT_DIR / "external_v1_paired_error_by_class.csv", index=False)
display(paired_class_df)

In [ ]:
# 4. Controlled 224x224 Resolution & Lesion-ROI Cropping Experiment (Testing the Bacterial Spot Hypothesis)
# We test 4 spatial scaling / cropping regimes on both External Field v1 (n=62) and Verified Field Dev (n=92):
#   1) 1.00x_full_scene (Standard 224x224 resize)
#   2) 0.70x_center_crop (1.43x linear zoom -> 2.04x lesion pixel area before 224x224)
#   3) 0.50x_closeup_crop (2.00x linear zoom -> 4.00x lesion pixel area before 224x224)
#   4) highres_448_5crop (Resize to 448x448 and average softmax across five 224x224 crops: 4 corners + center)

def predict_highres_5crop(model, norm_tf, pil_images: list[Image.Image], labels: list[int], batch_size: int = 16):
    five_crop_tf = transforms.Compose([
        transforms.Resize((448, 448), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.FiveCrop(224),
    ])
    all_probs = []
    with torch.inference_mode():
        for i in range(0, len(pil_images), batch_size):
            batch_imgs = pil_images[i : i + batch_size]
            crops_tensors = []
            for im in batch_imgs:
                crops = five_crop_tf(im)
                crops_tensors.append(torch.stack([norm_tf(c) for c in crops]))  # (5, 3, 224, 224)
            stacked = torch.stack(crops_tensors).to(DEVICE)  # (B, 5, 3, 224, 224)
            b, n_c, c, h, w = stacked.shape
            logits = model(stacked.view(b * n_c, c, h, w)).view(b, n_c, -1)
            probs = torch.softmax(logits.float(), dim=-1).mean(dim=1)  # average probability across 5 high-res crops
            all_probs.append(probs.cpu().numpy())
    probs_arr = np.vstack(all_probs)
    y_true = np.asarray(labels, dtype=int)
    y_pred = probs_arr.argmax(axis=1)
    conf = probs_arr.max(axis=1)
    return y_true, y_pred, conf, probs_arr


# Pre-load PIL images for External v1 (n=62) and Field Dev (n=92)
ext_pil_images, ext_labels = [], []
for r in audited_ext_df.itertuples(index=False):
    with Image.open(EXTERNAL_DIR / r.image_path) as im:
        ext_pil_images.append(im.convert("RGB"))
    ext_labels.append(int(r.label_id))

fdev_pil_images, fdev_labels = [], []
for idx in range(len(base_field_dev_ds)):
    row = base_field_dev_ds.metadata.iloc[idx]
    rec = base_field_dev_ds.source_datasets[row["source"]][int(row["row_id"])]
    fdev_pil_images.append(rec["image"].convert("RGB"))
    fdev_labels.append(int(row["label_id"]))

BACT_ID = LABEL_MAP["Bacterial Spot"]
SEPT_ID = LABEL_MAP["Septoria Leaf Spot"]

resolution_rows = []
for part_name, is_ext in (("external_field_v1", True), ("verified_field_dev", False)):
    for model_name in ("ResNet18 · Geometric + Mixed", "ViT-Small · Mixed", "ResNet18 · Lab baseline", "ViT-Small · Lab baseline"):
        model, tf, norm_tf = LOADED_MODELS[model_name]
        bs = CHECKPOINT_SPECS[model_name]["batch_size"]

        for regime_name, crop_frac in (("1.00x_full_scene", 1.0), ("0.70x_center_crop", 0.70), ("0.50x_closeup_crop", 0.50), ("highres_448_5crop", None)):
            if crop_frac is not None:
                if is_ext:
                    ds = ExternalManifestDataset(audited_ext_df, EXTERNAL_DIR, transform=tf, center_crop_fraction=crop_frac)
                else:
                    ds = CroppedPlantDiseaseDataset(base_field_dev_ds, transform=tf, center_crop_fraction=crop_frac)
                loader = DataLoader(ds, batch_size=bs, shuffle=False)
                yt, yp, cf, pr = run_inference(model, loader)
            else:
                if is_ext:
                    yt, yp, cf, pr = predict_highres_5crop(model, norm_tf, ext_pil_images, ext_labels, batch_size=16)
                else:
                    yt, yp, cf, pr = predict_highres_5crop(model, norm_tf, fdev_pil_images, fdev_labels, batch_size=16)

            bact_mask = (yt == BACT_ID)
            bact_n = int(np.sum(bact_mask))
            bact_correct = int(np.sum((yp == BACT_ID) & bact_mask))
            bact_as_sept = int(np.sum((yp == SEPT_ID) & bact_mask))
            bact_mean_true_prob = float(np.mean(pr[bact_mask, BACT_ID])) if bact_n > 0 else np.nan
            bact_mean_sept_prob = float(np.mean(pr[bact_mask, SEPT_ID])) if bact_n > 0 else np.nan

            resolution_rows.append({
                "partition": part_name,
                "model": model_name,
                "regime": regime_name,
                "overall_n": int(len(yt)),
                "overall_accuracy": round(float(accuracy_score(yt, yp)), 4),
                "overall_macro_f1": round(float(f1_score(yt, yp, labels=LABEL_IDS, average="macro", zero_division=0)), 4),
                "bacterial_spot_n": bact_n,
                "bacterial_spot_correct": bact_correct,
                "bacterial_spot_recall": round(bact_correct / bact_n, 4) if bact_n > 0 else np.nan,
                "bacterial_spot_misclassified_as_septoria": bact_as_sept,
                "bacterial_spot_mean_true_prob": round(bact_mean_true_prob, 4),
                "bacterial_spot_mean_septoria_prob": round(bact_mean_sept_prob, 4),
            })

resolution_df = pd.DataFrame(resolution_rows)
resolution_df.to_csv(OUTPUT_DIR / "bacterial_spot_resolution_experiment.csv", index=False)
print("=== 224x224 Resolution & Cropping Hypothesis Test Results ===")
display(resolution_df)

In [ ]:
# 5. Generate Publication Figures & Downloadable ZIP Bundle
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ext_sum = summary_df[summary_df["partition"] == "external_field_v1"].reset_index(drop=True)
ft_sum = summary_df[summary_df["partition"] == "plantdoc_field_test"].reset_index(drop=True)
x = np.arange(len(ext_sum))
w = 0.36
short_labels = ["ResNet18\nBaseline", "ViT-Small\nBaseline", "ResNet18\nGeom+Mixed", "ViT-Small\nMixed"]

axes[0].bar(x - w/2, ft_sum["macro_f1"], width=w, label="PlantDoc Field Test (n=47)", color="#4C72B0")
axes[0].bar(x + w/2, ext_sum["macro_f1"], width=w, label="External Field Eval v1 (n=62)", color="#DD8452")
axes[0].set_xticks(x)
axes[0].set_xticklabels(short_labels)
axes[0].set_ylim(0, 0.75)
axes[0].set_ylabel("Macro-F1")
axes[0].set_title("Confirmatory Field Macro-F1 Across Locked Models")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(x - w/2, ft_sum["accuracy"], width=w, label="PlantDoc Field Test (n=47)", color="#4C72B0")
axes[1].bar(x + w/2, ext_sum["accuracy"], width=w, label="External Field Eval v1 (n=62)", color="#DD8452")
axes[1].set_xticks(x)
axes[1].set_xticklabels(short_labels)
axes[1].set_ylim(0, 0.75)
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Confirmatory Field Accuracy Across Locked Models")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confirmatory_field_4models.png", dpi=200)
plt.show()

# Package all outputs into a single zip file and trigger download
zip_base = Path("/content/external_confirmatory_eval_outputs")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(OUTPUT_DIR))
print("Created output bundle:", zip_path)
files.download(zip_path)